## Generation of Protein Vector Given a 2D Sequence

Import Packages:

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import esm


Load the dataset:

In [2]:
df = pd.read_csv("Datasets/Fluorescent-Protein-Database.csv")

Generate t-scale for protein sequences:

In [3]:
tsc_df = pd.read_csv("Datasets/T_scales_table.csv")

tsc_df.columns = tsc_df.columns.str.strip()
tsc_df["symbol"] = tsc_df["symbol"].str.strip()

T_SCALE = {}

for _, row in tsc_df.iterrows():
    aa = row["symbol"]
    T_SCALE[aa] = [
        float(row["T_1"]),
        float(row["T_2"]),
        float(row["T_3"]),
        float(row["T_4"]),
        float(row["T_5"])
    ]

def sequence_to_tscales(seq):
    vecs = []
    for aa in seq:
        vecs.append(T_SCALE.get(aa, [0,0,0,0,0]))
    return np.array(vecs)

def tscale_features(seq):
    mat = sequence_to_tscales(seq)

    mean = mat.mean(axis=0)
    std = mat.std(axis=0)
    maxv = mat.max(axis=0)
    minv = mat.min(axis=0)

    return np.concatenate([mean, std, maxv, minv])

df["tscales"] = df["Protein sequence"].apply(tscale_features)

Using Evolutionary Scale Modeling (ESM-2) Embeddings

In [4]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

In [5]:
def get_esm_embedding(seq):
    data = [("protein", seq)]

    _, _, tokens = batch_converter(data)

    with torch.no_grad():
        results = model(tokens, repr_layers=[33])

    # layer 33 = final representation
    reps = results["representations"][33]

    # remove special tokens and mean pool
    embedding = reps[0, 1:-1].mean(0)

    return embedding.numpy()


df["esm"] = df["Protein sequence"].apply(get_esm_embedding)

Encode SMILES of chromophores

Split train/test

In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Normalize predictors and apply same normalization to the test dataset

In [5]:
scaler = StandardScaler()

num_cols = ["Stokes shift", "kDa"] # Which columns to normalize

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])